In [1]:
import transformers
print(f"Transformers version: {transformers.__version__}")

try:
    from transformers import TFAutoModelForQuestionAnswering
    print("TFAutoModelForQuestionAnswering est disponible.")
except ImportError as e:
    print(f"TFAutoModelForQuestionAnswering N'EST PAS disponible. Erreur: {e}")

Transformers version: 4.38.2
TFAutoModelForQuestionAnswering est disponible.


In [2]:
import transformers
print(f"Transformers version: {transformers.__version__}")

try:
    from transformers import TFAutoModelForQuestionAnswering
    print("TFAutoModelForQuestionAnswering est disponible.")
except ImportError as e:
    print(f"TFAutoModelForQuestionAnswering N'EST PAS disponible. Erreur: {e}")

Transformers version: 4.38.2
TFAutoModelForQuestionAnswering est disponible.


#  Que peuvent faire les *transformers* ?

Installez la bibliothèque 🤗 *Transformers* pour exécuter ce *notebook*.

In [3]:
!pip install transformers[sentencepiece]

In [4]:
from transformers import pipeline

### Analyse de sentiments

In [5]:
classifier = pipeline("sentiment-analysis")
classifier("I've been waiting for a HuggingFace course my whole life.")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision af0f99b (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authe

[{'label': 'POSITIVE', 'score': 0.9598049521446228}]

Intéressant ! On observe que le résultat est négatif là où pour la version en anglais le résultat est positif.

In [6]:
classifier(
    ["J'ai attendu un cours d'HuggingFace toute ma vie.",
     "Je déteste tellement ça !"]
) # pour classifier plusieurs phrases

[{'label': 'POSITIVE', 'score': 0.6333001255989075},
 {'label': 'NEGATIVE', 'score': 0.8847918510437012}]

La phrase "J'ai attendu un cours d'HuggingFace toute ma vie." qui était précedemment négative devient à présent positive.

### Zéro shot classification

In [8]:
classifier = pipeline("zero-shot-classification", model="BaptisteDoyen/camembert-base-xnli")
classifier(
    "C'est un cours sur la bibliothèque Transformers",
    candidate_labels=["éducation", "politique", "affaires"],
)

OSError: BaptisteDoyen/camembert-base-xnli does not appear to have a file named config.json. Checkout 'https://huggingface.co/BaptisteDoyen/camembert-base-xnli/main' for available files.

### Génération de texte

In [9]:
generator = pipeline("text-generation", model="asi/gpt-fr-cased-small")
generator("# Dans ce cours, nous vous enseignerons comment")

/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1178: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


[{'generated_text': '# Dans ce cours, nous vous enseignerons comment nous pouvons être des êtres humains. " " Nous'}]

### Remplacement des mots masqués

In [10]:
unmasker = pipeline("fill-mask", model="camembert-base")
unmasker(" Ce cours vous apprendra tout sur les modèles <mask>.", top_k=2)

Some weights of the model checkpoint at camembert-base were not used when initializing CamembertForMaskedLM: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing CamembertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CamembertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[{'score': 0.10731794685125351,
  'token': 15328,
  'token_str': 'automobiles',
  'sequence': 'Ce cours vous apprendra tout sur les modèles automobiles.'},
 {'score': 0.0752047672867775,
  'token': 4007,
  'token_str': 'électriques',
  'sequence': 'Ce cours vous apprendra tout sur les modèles électriques.'}]

### Reconnaissance d'entités nommées

In [11]:
ner = pipeline("ner", model="Jean-Baptiste/camembert-ner", grouped_entities=True)
ner("Je m'appelle Sylvain et je travaille à Hugging Face à Brooklyn.")

/usr/local/lib/python3.13/dist-packages/transformers/pipelines/token_classification.py:168: UserWarning: `grouped_entities` is deprecated and will be removed in version v5.0.0, defaulted to `aggregation_strategy="AggregationStrategy.SIMPLE"` instead.
  warnings.warn(


[{'entity_group': 'PER',
  'score': np.float32(0.6074448),
  'word': 'Sylvain',
  'start': 12,
  'end': 20},
 {'entity_group': 'LOC',
  'score': np.float32(0.6868436),
  'word': 'Hugging Face',
  'start': 38,
  'end': 51},
 {'entity_group': 'LOC',
  'score': np.float32(0.99523795),
  'word': 'Brooklyn',
  'start': 53,
  'end': 62}]

### Réponse à des questions

In [15]:
# @title
#!pip install --upgrade transformers==4.38.2
#!pip install --upgrade tensorflow

import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModelForQuestionAnswering, pipeline

# Load tokenizer and model explicitly
tokenizer = AutoTokenizer.from_pretrained("etalab-ia/camembert-base-squadFR-fquad-piaf")
model = TFAutoModelForQuestionAnswering.from_pretrained("etalab-ia/camembert-base-squadFR-fquad-piaf")

# Pass the loaded tokenizer and model to the pipeline
question_answerer = pipeline("question-answering", model=model, tokenizer=tokenizer)
question_answerer(
    question="Où est-ce que je travaille ?",
    context="Je m'appelle Sylvain et je travaille à Hugging Face à Brooklyn.",
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
All PyTorch model weights were used when initializing TFCamembertForQuestionAnswering.

All the weights of TFCamembertForQuestionAnswering were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFCamembertForQuestionAnswering for predictions without further training.


{'score': 0.5418347120285034,
 'start': 38,
 'end': 63,
 'answer': ' Hugging Face à Brooklyn.'}

In [14]:
import transformers
print(f"Transformers version: {transformers.__version__}")

try:
    from transformers import TFAutoModelForQuestionAnswering
    print("TFAutoModelForQuestionAnswering est disponible.")
except ImportError as e:
    print(f"TFAutoModelForQuestionAnswering N'EST PAS disponible. Erreur: {e}")

Transformers version: 4.38.2
TFAutoModelForQuestionAnswering est disponible.


###  Résumé

In [ ]:
summarizer = pipeline("summarization", model="moussaKam/barthez-orangesum-abstract")
summarizer(
    """
    L'Amérique a changé de façon spectaculaire au cours des dernières années. Non seulement le nombre de
    diplômés dans les disciplines traditionnelles de l'ingénierie telles que le génie mécanique, civil,
    l'électricité, la chimie et l'aéronautique a diminué, mais dans la plupart
    des grandes universités américaines, les programmes d'études d'ingénierie se concentrent désormais sur
    et encouragent largement l'étude des sciences de l'ingénieur. Par conséquent, il y a
    de moins en moins d'offres dans les sujets d'ingénierie traitant de l'infrastructure,
    l'environnement et les questions connexes, et une plus grande concentration sur les sujets de haute
    technologie, qui soutiennent en grande partie des développements scientifiques de plus en plus
    complexes. Si cette dernière est importante, elle ne doit pas se faire au détriment
    de l'ingénierie plus traditionnelle.

    Les économies en développement rapide telles que la Chine et l'Inde, ainsi que d'autres
    pays industrialisés d'Europe et d'Asie, continuent d'encourager et de promouvoir
    l'enseignement de l'ingénierie. La Chine et l'Inde, respectivement, diplôment
    six et huit fois plus d'ingénieurs traditionnels que les États-Unis.
    Les autres pays industriels maintiennent au minimum leur production, tandis que l'Amérique
    souffre d'une baisse de plus en plus importante du nombre de diplômés en ingénierie
    et un manque d'ingénieurs bien formés.
"""
)

###  Traduction

In [16]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-en-fr")
translator("This course is produced by Hugging Face.")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:197: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


[{'translation_text': 'Ce cours est produit par Hugging Face.'}]